# Module 2 - Data Quality Enhancement

## Input:
university_raw_data.csv

## Output:
university_cleaned.csv

## Objective:
Improve dataset quality through rule-based recovery, interpolation, statistical imputation, and machine learning.

This notebook improves the quality of the cleaned higher education dataset using statistical and machine learning based imputation techniques.

Goals

- Reduce missing values below 2%
- Preserve historical consistency
- Improve data completeness
- Generate a Tableau / Power BI ready dataset

Dataset Timeline

2017–2026

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.impute import KNNImputer
from sklearn.ensemble import RandomForestRegressor

pd.set_option("display.max_columns", None)

In [2]:
# ==========================================
# Paths
# ==========================================

SCRIPT_DIR = Path.cwd().parent.parent

INPUT_PATH = (
    SCRIPT_DIR /
    "datasets" /
    "final" /
    "Module_1_Deliverables" /
    "university_raw_data.csv"
)

OUTPUT_PATH = (
    SCRIPT_DIR /
    "datasets" /
    "final" /
    "Module_2_Deliverables"
)

OUTPUT_FILE = (
    OUTPUT_PATH /
    "university_cleaned.csv"
)

In [3]:
df = pd.read_csv(
    INPUT_PATH,
    low_memory=False
)

print(df.shape)

df.head()

(23263, 36)


,University,Country,Country Code,City,Region,University Type,University Age Band,Academic Reputation Score,Employer Reputation Score,Faculty Student Score,Citations per Faculty Score,International Faculty Score,International Students Score,International Research Network Score,Employment Outcomes Score,Sustainability Score,Faculty Count,Research Output,Year,Student Population,Teaching,Research Environment,Research Quality,Industry Impact,International Outlook,Publications,Citation Count,h-index,Research Productivity Index,Subject Area,Female Student Percentage,Male Student Percentage,Overall Score,Students to Staff Ratio,International Students,Continent
0,Don State Technical University,Russia,RU,Rostov-on-Don,Europe,Public,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"1,860",High,2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Life Sciences & Medicine,NaN,NaN,NaN,14.0,"2,627",Europe
1,Don State Technical University,Russia,RU,Rostov-on-Don,Europe,Public,4.0,3.7,3.6,12.7,1.4,1.6,15.7,12.2,3.4,NaN,NaN,NaN,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Life Sciences & Medicine,NaN,NaN,NaN,NaN,NaN,Europe
2,Don State Technical University,Russia,RU,Rostov-on-Don,Europe,Public,4.0,4.2,3.1,13.3,1.4,1.6,14.8,1.0,7.2,1.3,NaN,NaN,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Life Sciences & Medicine,NaN,NaN,NaN,NaN,NaN,Europe
3,Don State Technical University,Russia,RU,Rostov-on-Don,Europe,Public,4.0,3.5,3.5,12.8,1.4,1.6,14.9,11.4,1.9,1.0,NaN,NaN,2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Life Sciences & Medicine,NaN,NaN,NaN,NaN,NaN,Europe
4,ULACIT - Universidad Latinoamericana de Cienc...,Costa Rica,CR,San José,Latin America,Private,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,218,Low,2017,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Arts & Humanities,NaN,NaN,NaN,14.0,NaN,North America


In [4]:
print("="*70)
print("INITIAL DATA QUALITY")
print("="*70)

print(f"Rows    : {len(df):,}")
print(f"Columns : {len(df.columns)}")

missing_total = df.isna().sum().sum()

total_cells = df.shape[0] * df.shape[1]

missing_percent = (
    missing_total /
    total_cells *
    100
)

print(f"\nTotal Missing Values : {missing_total:,}")

print(f"Dataset Completeness : {100-missing_percent:.2f}%")

INITIAL DATA QUALITY
Rows    : 23,263
Columns : 36

Total Missing Values : 286,961
Dataset Completeness : 65.73%


In [5]:
initial_missing = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .to_frame("Missing Values")
)

initial_missing["Missing %"] = (
    initial_missing["Missing Values"] /
    len(df) *
    100
).round(2)

display(initial_missing)

,Missing Values,Missing %
Sustainability Score,18903,81.26
International Faculty Score,17748,76.29
International Students Score,17569,75.52
International Research Network Score,17392,74.76
Employment Outcomes Score,17375,74.69
Citations per Faculty Score,17366,74.65
Faculty Student Score,17363,74.64
Employer Reputation Score,17339,74.53
Faculty Count,16858,72.47
Research Output,16782,72.14


In [6]:
important_columns = [

    "Academic Reputation Score",
    "Employer Reputation Score",
    "Faculty Student Score",
    "Citations per Faculty Score",
    "International Faculty Score",
    "International Students Score",
    "International Research Network Score",
    "Employment Outcomes Score",
    "Sustainability Score",

    "Teaching",
    "Research Environment",
    "Research Quality",
    "Industry Impact",
    "International Outlook"
]

availability = (
    df.groupby("Year")[important_columns]
      .apply(lambda x: x.notna().sum())
)

display(availability)

,Academic Reputation Score,Employer Reputation Score,Faculty Student Score,Citations per Faculty Score,International Faculty Score,International Students Score,International Research Network Score,Employment Outcomes Score,Sustainability Score,Teaching,Research Environment,Research Quality,Industry Impact,International Outlook
Year,,,,,,,,,,,,,,
2017,404,0,0,0,0,0,0,0,0,981,981,981,981,981
2018,403,0,0,0,0,0,0,0,0,1103,1103,1103,1103,1103
2019,505,0,0,0,0,0,0,0,0,1258,1258,1258,1258,1258
2020,501,0,0,0,0,0,0,0,0,1397,1397,1397,1397,1397
2021,506,0,0,0,0,0,0,0,0,1526,1526,1526,1526,1526
2022,501,0,0,0,0,0,0,0,0,1662,1662,1662,1662,1662
2023,1423,1422,1421,1418,1325,1366,1410,1411,0,1799,1799,1799,1799,1799
2024,1498,1497,1474,1474,1372,1418,1494,1474,1398,1904,1904,1904,1904,1904
2025,1504,1504,1504,1504,1404,1446,1503,1504,1485,2092,2092,2092,2092,2092


In [7]:
# ==========================================
# Column Categories
# ==========================================

static_columns = [

    "Country",
    "Country Code",
    "Region",
    "Continent",
    "City",
    "University Type",
    "Subject Area"

]

time_series_columns = [

    "Student Population",
    "Faculty Count",
    "Students to Staff Ratio",
    "International Students",
    "Female Student Percentage",
    "Male Student Percentage"

]

research_columns = [

    "Publications",
    "Citation Count",
    "h-index",
    "Research Productivity Index"

]

print("Static :", len(static_columns))
print("Time :", len(time_series_columns))
print("Research :", len(research_columns))

Static : 7
Time : 6
Research : 4


In [8]:
# ==========================================
# Static Value Recovery
# ==========================================

for column in static_columns:

    df[column] = (

        df.groupby("University")[column]

          .transform(

              lambda x: x.ffill().bfill()

          )

    )

print("Static recovery completed.")

Static recovery completed.


In [9]:
before = initial_missing.copy()

after = (

    df[static_columns]

      .isna()

      .sum()

      .to_frame("Remaining Missing")

)

display(after)

,Remaining Missing
Country,0
Country Code,4
Region,0
Continent,0
City,11387
University Type,395
Subject Area,6


In [10]:
# ==========================================
# Country-wise Categorical Recovery
# ==========================================

categorical_columns = [
    "University Type",
    "Subject Area"
]

for column in categorical_columns:

    mode_lookup = (
        df.groupby("Country")[column]
          .agg(
              lambda x: x.mode().iloc[0]
              if not x.mode().empty
              else np.nan
          )
    )

    df[column] = (
        df[column]
          .fillna(df["Country"].map(mode_lookup))
    )

print("Country-wise categorical recovery completed.")

Country-wise categorical recovery completed.


In [11]:
# ==========================================
# Recovery Report
# ==========================================

report = (
    df[
        [
            "Country",
            "Country Code",
            "Region",
            "Continent",
            "City",
            "University Type",
            "Subject Area"
        ]
    ]
    .isna()
    .sum()
    .to_frame("Remaining Missing")
)

display(report)

,Remaining Missing
Country,0
Country Code,4
Region,0
Continent,0
City,11387
University Type,395
Subject Area,6


In [12]:
# ==========================================
# Sort Dataset
# ==========================================

df = (
    df
    .sort_values(
        ["University", "Year"]
    )
    .reset_index(drop=True)
)

In [13]:
# ==========================================
# University-wise Interpolation
# ==========================================

time_columns = [

    "Student Population",
    "Faculty Count",
    "Students to Staff Ratio",
    "International Students",
    "Female Student Percentage",
    "Male Student Percentage"

]

for column in time_columns:

    df[column] = (
        df.groupby("University")[column]
          .transform(
              lambda x: (
                  pd.to_numeric(
                      x.astype(str)
                       .str.replace(",", "", regex=False)
                       .str.replace("%", "", regex=False)
                       .str.strip(),
                      errors="coerce"
                  ).interpolate(method="linear", limit_direction="both")
              )
          )
    )

print("Interpolation completed.")

Interpolation completed.


In [14]:
# ==========================================
# Time-Series Recovery Report
# ==========================================

report = (

    df[time_columns]

      .isna()

      .sum()

      .to_frame("Remaining Missing")

)

display(report)

,Remaining Missing
Student Population,5956
Faculty Count,11095
Students to Staff Ratio,895
International Students,1017
Female Student Percentage,2911
Male Student Percentage,2911


In [15]:
# ==========================================
# Missing Values by Year
# ==========================================

missing_by_year = (
    df.groupby("Year")
      .apply(lambda x: x.isna().sum().sum())
      .to_frame("Missing Values")
)

missing_by_year["Completeness (%)"] = (
    (
        1
        - missing_by_year["Missing Values"]
        / (df.shape[1] * df.groupby("Year").size())
    )
    * 100
).round(2)

display(missing_by_year)

,Missing Values,Completeness (%)
Year,,
2017,20500,62.73
2018,22514,62.66
2019,24714,62.85
2020,26808,62.86
2021,29736,62.62
2022,32747,62.43
2023,27252,71.43
2024,27789,72.47
2025,30083,72.01


In [16]:
# ==========================================
# Top Missing Columns
# ==========================================

missing = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .to_frame("Missing")
)

missing["Missing %"] = (
    missing["Missing"] / len(df) * 100
).round(2)

display(missing.head(20))

,Missing,Missing %
Sustainability Score,18903,81.26
International Faculty Score,17748,76.29
International Students Score,17569,75.52
International Research Network Score,17392,74.76
Employment Outcomes Score,17375,74.69
Citations per Faculty Score,17366,74.65
Faculty Student Score,17363,74.64
Employer Reputation Score,17339,74.53
Research Output,16782,72.14
Academic Reputation Score,14517,62.40


In [17]:
# ==========================================
# QS Metrics Missing Values
# ==========================================

qs_metrics = [

    "Academic Reputation Score",
    "Employer Reputation Score",
    "Faculty Student Score",
    "Citations per Faculty Score",
    "International Faculty Score",
    "International Students Score",
    "International Research Network Score",
    "Employment Outcomes Score",
    "Sustainability Score"

]

qs_missing = (

    df[qs_metrics]

      .isna()

      .sum()

      .to_frame("Missing")

)

qs_missing["Missing %"] = (

    qs_missing["Missing"]

    / len(df)

    * 100

).round(2)

display(qs_missing)

,Missing,Missing %
Academic Reputation Score,14517,62.40
Employer Reputation Score,17339,74.53
Faculty Student Score,17363,74.64
Citations per Faculty Score,17366,74.65
International Faculty Score,17748,76.29
International Students Score,17569,75.52
International Research Network Score,17392,74.76
Employment Outcomes Score,17375,74.69
Sustainability Score,18903,81.26


In [18]:
# ==========================================
# Research Metrics
# ==========================================

research_metrics = [

    "Publications",
    "Citation Count",
    "h-index",
    "Research Productivity Index"

]

display(

    df[research_metrics]

      .isna()

      .sum()

)

Publications                   2911
Citation Count                 2911
h-index                        2911
Research Productivity Index    2911
dtype: int64

In [19]:
# ==========================================
# Recovery Candidates
# ==========================================

candidate_columns = [

    "Student Population",
    "Faculty Count",
    "Students to Staff Ratio",
    "International Students",
    "Female Student Percentage",
    "Male Student Percentage",
    "University Type",
    "Subject Area",
    "City"

]

candidate_missing = (

    df[candidate_columns]

      .isna()

      .sum()

      .to_frame("Missing")

)

display(candidate_missing)

,Missing
Student Population,5956
Faculty Count,11095
Students to Staff Ratio,895
International Students,1017
Female Student Percentage,2911
Male Student Percentage,2911
University Type,395
Subject Area,6
City,11387


In [20]:
# ==========================================
# Country-wise Numeric Imputation
# ==========================================

numeric_columns = [
    "Student Population",
    "Faculty Count",
    "Students to Staff Ratio",
    "International Students",
    "Female Student Percentage",
    "Male Student Percentage"
]

for column in numeric_columns:

    country_median = (
        df.groupby("Country")[column]
          .transform("median")
    )

    df[column] = df[column].fillna(country_median)

print("Country-wise numeric imputation completed.")

Country-wise numeric imputation completed.


In [21]:
# ==========================================
# Global Median Fallback
# ==========================================

for column in numeric_columns:
    df[column] = df[column].fillna(df[column].median())

print("Global fallback completed.")

Global fallback completed.


In [22]:
remaining = (
    df[numeric_columns]
      .isna()
      .sum()
      .to_frame("Remaining Missing")
)

display(remaining)

,Remaining Missing
Student Population,0
Faculty Count,0
Students to Staff Ratio,0
International Students,0
Female Student Percentage,0
Male Student Percentage,0


In [23]:
# ==========================================
# Enable Experimental Iterative Imputer
# ==========================================

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

In [24]:
# ==========================================
# Features for Imputation
# ==========================================

imputation_columns = [

    "Overall Score",

    "Teaching",
    "Research Environment",
    "Research Quality",
    "Industry Impact",
    "International Outlook",

    "Publications",
    "Citation Count",
    "h-index",
    "Research Productivity Index",

    "Academic Reputation Score",
    "Employer Reputation Score",
    "Faculty Student Score",
    "Citations per Faculty Score",
    "International Faculty Score",
    "International Students Score",
    "International Research Network Score",
    "Employment Outcomes Score",
    "Sustainability Score"

]

In [25]:
# ==========================================
# Year-wise Iterative Imputation
# ==========================================

for year in sorted(df["Year"].unique()):

    year_mask = df["Year"] == year

    year_data = df.loc[
        year_mask,
        imputation_columns
    ]

    if year_data.isna().sum().sum() == 0:
        continue

    imputer = IterativeImputer(
        random_state=42,
        max_iter=20
    )

    # ensure numeric inputs and preserve index/columns for assignment
    year_numeric = year_data.apply(
        lambda col: pd.to_numeric(
            col.astype(str)
               .str.replace(",", "", regex=False)
               .str.replace("%", "", regex=False)
               .str.strip(),
            errors="coerce"
        )
    )

    imputed = imputer.fit_transform(year_numeric)

    # Get columns that had data (non-all-NaN)
    valid_cols = year_numeric.columns[year_numeric.notna().any()].tolist()
    
    # Create DataFrame with imputed values using only valid columns
    imputed_df = pd.DataFrame(
        imputed,
        index=year_data.index,
        columns=valid_cols
    )
    
    # Assign back to dataframe
    # Align and assign imputed DataFrame directly to preserve index/column alignment
    # ensure target columns in the main df are numeric (so float assignment is allowed)
    df[valid_cols] = df[valid_cols].apply(
        lambda col: pd.to_numeric(
            col.astype(str)
               .str.replace(",", "", regex=False)
               .str.replace("%", "", regex=False)
               .str.strip(),
            errors="coerce"
        )
    )

    # assign imputed values aligning by index and columns
    df.loc[year_data.index, valid_cols] = imputed_df.astype(float).values

print("Year-wise MICE imputation completed.")

C:\Users\shashank\AppData\Roaming\Python\Python312\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: [11 12 13 14 15 16 17 18]. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
C:\Users\shashank\AppData\Roaming\Python\Python312\site-packages\sklearn\impute\_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(
C:\Users\shashank\AppData\Roaming\Python\Python312\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: [11 12 13 14 15 16 17 18]. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
C:\Users\shashank\AppData\Roaming\Python\Python312\site-packages\sklearn\impute\_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(
C:\Users\shashank\AppData\Roaming\Python\Python312\site-packag

Year-wise MICE imputation completed.


In [26]:
# ==========================================
# Remaining Missing Values
# ==========================================

remaining = (
    df[imputation_columns]
      .isna()
      .sum()
      .sort_values(ascending=False)
      .to_frame("Remaining Missing")
)

display(remaining)

,Remaining Missing
Sustainability Score,14337
Employment Outcomes Score,11687
International Students Score,11687
Faculty Student Score,11687
Citations per Faculty Score,11687
Employer Reputation Score,11687
International Research Network Score,11687
International Faculty Score,11687
Research Environment,0
Overall Score,0


In [27]:
# ==========================================
# Random Forest Imputation
# ==========================================

from sklearn.ensemble import RandomForestRegressor

In [28]:
# ==========================================
# Features Used For Prediction
# ==========================================

feature_columns = [

    "Overall Score",

    "Teaching",
    "Research Environment",
    "Research Quality",
    "Industry Impact",
    "International Outlook",

    "Publications",
    "Citation Count",
    "h-index",
    "Research Productivity Index",

    "Student Population",
    "Faculty Count",
    "Students to Staff Ratio",
    "International Students"

]

In [29]:
# ==========================================
# Target Columns
# ==========================================

target_columns = [

    "Academic Reputation Score",
    "Employer Reputation Score",
    "Faculty Student Score",
    "Citations per Faculty Score",
    "International Faculty Score",
    "International Students Score",
    "International Research Network Score",
    "Employment Outcomes Score",
    "Sustainability Score"

]

In [30]:
# ==========================================
# Random Forest Imputation
# ==========================================

for target in target_columns:

    train = df[df[target].notna()].copy()

    predict = df[df[target].isna()].copy()

    if len(predict) == 0:
        continue

    train = train.dropna(subset=feature_columns)

    predict = predict.dropna(subset=feature_columns)

    if len(train) < 100:
        continue

    model = RandomForestRegressor(

        n_estimators=300,

        random_state=42,

        n_jobs=-1

    )

    model.fit(

        train[feature_columns],

        train[target]

    )

    predicted = model.predict(

        predict[feature_columns]

    )

    df.loc[
        predict.index,
        target
    ] = predicted

    print(f"{target:<45} Filled : {len(predicted)}")

Employer Reputation Score                     Filled : 11687
Faculty Student Score                         Filled : 11687
Citations per Faculty Score                   Filled : 11687
International Faculty Score                   Filled : 11687
International Students Score                  Filled : 11687
International Research Network Score          Filled : 11687
Employment Outcomes Score                     Filled : 11687
Sustainability Score                          Filled : 14337


In [31]:
# ==========================================
# Remaining Missing
# ==========================================

remaining = (

    df[target_columns]

      .isna()

      .sum()

      .sort_values()

      .to_frame("Remaining Missing")

)

display(remaining)

,Remaining Missing
Academic Reputation Score,0
Employer Reputation Score,0
Faculty Student Score,0
Citations per Faculty Score,0
International Faculty Score,0
International Students Score,0
International Research Network Score,0
Employment Outcomes Score,0
Sustainability Score,0


In [32]:
# ==========================================
# Overall Dataset Quality
# ==========================================

missing_total = df.isna().sum().sum()

total_cells = df.shape[0] * df.shape[1]

completeness = (
    100 -
    (
        missing_total /
        total_cells
    ) * 100
)

print("="*70)

print(f"Missing Values : {missing_total:,}")

print(f"Completeness : {completeness:.2f}%")

print("="*70)

Missing Values : 39,173
Completeness : 95.32%


In [33]:
# ==========================================
# University-wise Metadata Recovery
# ==========================================

metadata_columns = [

    "City",
    "University Type",
    "Research Output",
    "University Age Band",
    "Subject Area"

]

for column in metadata_columns:

    if column not in df.columns:
        continue

    df[column] = (

        df.groupby("University")[column]

          .transform(

              lambda x: x.ffill().bfill()

          )

    )

print("Metadata recovery completed.")

Metadata recovery completed.


In [34]:
# ==========================================
# Country-wise Metadata Recovery
# ==========================================

for column in metadata_columns:

    if column not in df.columns:
        continue

    mode_lookup = (

        df.groupby("Country")[column]

          .agg(

              lambda x:

              x.mode().iloc[0]

              if not x.mode().empty

              else np.nan

          )

    )

    df[column] = (

        df[column]

          .fillna(

              df["Country"].map(mode_lookup)

          )

    )

print("Country metadata recovery completed.")

Country metadata recovery completed.


In [35]:
# ==========================================
# Remaining Metadata Missing Values
# ==========================================

existing_metadata_columns = [
    column for column in metadata_columns
    if column in df.columns
]

if existing_metadata_columns:
    display(
        df[existing_metadata_columns]
          .isna()
          .sum()
          .to_frame("Remaining Missing")
    )
else:
    print("No metadata columns found in the dataframe.")

,Remaining Missing
City,454
University Type,395
Research Output,395
University Age Band,250
Subject Area,6


In [36]:
# ==========================================
# Data Quality Audit
# ==========================================

numeric_cols = df.select_dtypes(include="number").columns

print("=" * 70)
print("DATA QUALITY AUDIT")
print("=" * 70)

for col in numeric_cols:
    print(f"\n{col}")
    print(f"  Min : {df[col].min()}")
    print(f"  Max : {df[col].max()}")
    print(f"  Missing : {df[col].isna().sum()}")

DATA QUALITY AUDIT

University Age Band
  Min : 1.0
  Max : 5.0
  Missing : 250

Academic Reputation Score
  Min : -271.5755348601168
  Max : 124.0869250611195
  Missing : 0

Employer Reputation Score
  Min : -13.1845438257561
  Max : 108.883895965168
  Missing : 0

Faculty Student Score
  Min : -85.6691051470306
  Max : 127.96362907987617
  Missing : 0

Citations per Faculty Score
  Min : -71.97845741350392
  Max : 135.03992455903997
  Missing : 0

International Faculty Score
  Min : -116.1914786720688
  Max : 111.9075316988426
  Missing : 0

International Students Score
  Min : -87.67119822387117
  Max : 100.0
  Missing : 0

International Research Network Score
  Min : -33.32612550871467
  Max : 130.24027333664682
  Missing : 0

Employment Outcomes Score
  Min : -3.0851495729478207
  Max : 126.94126982129394
  Missing : 0

Sustainability Score
  Min : -18.578398234580977
  Max : 115.33615173323972
  Missing : 0

Faculty Count
  Min : 1.0
  Max : 20311.0
  Missing : 0

Year
  Min : 20

In [37]:
# ==========================================
# Invalid Range Report
# ==========================================

score_columns = [
    "Academic Reputation Score",
    "Employer Reputation Score",
    "Faculty Student Score",
    "Citations per Faculty Score",
    "International Faculty Score",
    "International Students Score",
    "International Research Network Score",
    "Employment Outcomes Score",
    "Sustainability Score",
    "Teaching",
    "Research Environment",
    "Research Quality",
    "Industry Impact",
    "International Outlook"
]

for col in score_columns:
    if col in df.columns:
        invalid = df[(df[col] < 0) | (df[col] > 100)]

        if len(invalid) > 0:
            print(f"\n{'='*70}")
            print(f"{col}")
            print(f"Invalid values: {len(invalid)}")
            print(invalid[["University", col]].head(10))


Academic Reputation Score
Invalid values: 211
                                         University  Academic Reputation Score
89                Abdul Wali Khan University Mardan                  -1.359311
144                             Acıbadem University                  -1.007889
199                        Aichi Medical University                  -0.310984
222                                  Air University                  -2.437188
223                                  Air University                  -1.965335
480  American International University – Bangladesh                  -0.980727
520                   American University of Beirut                  -1.203982
604                    An-Najah National University                 -14.002040
606                    An-Najah National University                  -2.205081
611                              Anadolu University                -271.575535

Employer Reputation Score
Invalid values: 230
                                     

In [38]:
# ==========================================
# Final Data Validation
# ==========================================

# Score columns (0-100)
score_columns = [
    "Academic Reputation Score",
    "Employer Reputation Score",
    "Faculty Student Score",
    "Citations per Faculty Score",
    "International Faculty Score",
    "International Students Score",
    "International Research Network Score",
    "Employment Outcomes Score",
    "Sustainability Score",
    "Teaching",
    "Research Environment",
    "Research Quality",
    "Industry Impact",
    "International Outlook"
]

# Count columns (>=0)
count_columns = [
    "Student Population",
    "International Students",
    "Publications",
    "Citation Count",
    "h-index"
]

# Ratio column
ratio_columns = [
    "Students to Staff Ratio"
]

# Clip score columns
for col in score_columns:
    if col in df.columns:
        df[col] = df[col].clip(0, 100)

# Clip count columns
for col in count_columns:
    if col in df.columns:
        df[col] = df[col].clip(lower=0)

# Clip ratios
for col in ratio_columns:
    if col in df.columns:
        df[col] = df[col].clip(lower=0)

print("=" * 60)
print("DATA VALIDATION COMPLETED")
print("=" * 60)
print("✓ Scores clipped to 0-100")
print("✓ Counts clipped to >= 0")
print("✓ Ratios clipped to >= 0")

DATA VALIDATION COMPLETED
✓ Scores clipped to 0-100
✓ Counts clipped to >= 0
✓ Ratios clipped to >= 0


In [39]:
# ==========================================
# Top Remaining Missing Columns
# ==========================================

remaining = (

    df.isna()

      .sum()

      .sort_values(ascending=False)

      .to_frame("Missing")

)

remaining["Missing %"] = (

    remaining["Missing"]

    / len(df)

    * 100

).round(2)

display(

    remaining.head(25)

)

,Missing,Missing %
City,454,1.95
University Type,395,1.70
Research Output,395,1.70
University Age Band,250,1.07
Subject Area,6,0.03
Country Code,4,0.02
University,0,0.00
Country,0,0.00
Employer Reputation Score,0,0.00
Faculty Student Score,0,0.00


In [40]:
# ==========================================
# Export Final Dataset
# ==========================================

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("=" * 60)
print("MODULE 2 COMPLETED")
print("=" * 60)
print(f"Saved : {OUTPUT_FILE}")

MODULE 2 COMPLETED
Saved : c:\Users\shashank\Desktop\Infosys SpringBoard\higher-education-dv\datasets\final\Module_2_Deliverables\university_cleaned.csv


In [42]:
# ==========================================
# Final Module 2 Summary
# ==========================================

verify_df = df.copy()

missing = verify_df.isna().sum().sum()
cells = verify_df.shape[0] * verify_df.shape[1]

missing_percentage = (missing / cells) * 100
completeness = 100 - missing_percentage

negative_values = (
    verify_df.select_dtypes(include="number") < 0
).sum().sum()

print("=" * 70)
print("MODULE 2 SUMMARY")
print("=" * 70)

print(f"Rows                 : {len(verify_df):,}")
print(f"Columns              : {verify_df.shape[1]}")
print(f"Missing Values       : {missing:,}")
print(f"Missing Percentage   : {missing_percentage:.2f}%")
print(f"Completeness         : {completeness:.2f}%")
print(f"Negative Values      : {negative_values}")
print(f"Numeric Validation    : Passed")

MODULE 2 SUMMARY
Rows                 : 23,263
Columns              : 36
Missing Values       : 1,504
Missing Percentage   : 0.18%
Completeness         : 99.82%
Negative Values      : 0
Numeric Validation    : Passed
